Remove gaps from fasta file and edit gff accordingly.

In [ ]:
def remove_gaps_and_update_gff(input_fasta, input_gff, output_fasta=None, output_gff=None, save_fasta = True):
    def input_to_output_filename(filename):
        return filename[:-4]+"_nogaps"+filename[-4:]
    if not output_gff:
        output_gff = input_to_output_filename(input_gff)
    if not output_fasta:
        output_fasta = input_to_output_filename(input_fasta)
    # Step 1: Remove gaps from the fasta file
    fasta_sequences = {}
    with open(input_fasta, 'r') as f:
        current_id = None
        current_sequence = ""
        for line in f:
            if line.startswith('>'):
                if current_id:
                    fasta_sequences[current_id] = current_sequence
                current_id = line.strip()[1:]
                current_sequence = ""
            else:
                current_sequence += line.strip().replace('-', '')
        if current_id:
            fasta_sequences[current_id] = current_sequence

    # Step 2: Update the GFF file
    with open(input_gff, 'r') as f_in, open(output_gff, 'w') as f_out:
        for line in f_in:
            if line.startswith('#'):
                f_out.write(line)
            else:
                parts = line.strip().split('\t')
                sequence_id = parts[0]
                start_pos = int(parts[3])
                end_pos = int(parts[4])

                # Adjust positions based on gap removal
                if sequence_id in fasta_sequences:
                    sequence_length = len(fasta_sequences[sequence_id])
                    if start_pos <= sequence_length and end_pos <= sequence_length:
                        new_start = start_pos
                        new_end = end_pos
                        for pos in range(start_pos, end_pos + 1):
                            if fasta_sequences[sequence_id][pos - 1] == '-':
                                new_end -= 1
                                continue
                            else:
                                break
                        for pos in range(end_pos, start_pos - 1, -1):
                            if fasta_sequences[sequence_id][pos - 1] == '-':
                                new_start += 1
                            else:
                                break
                        if new_start <= new_end:
                            f_out.write(f"{sequence_id}\t{parts[1]}\t{parts[2]}\t{new_start}\t{new_end}\t{parts[5]}\t{parts[6]}\t{parts[7]}\t{parts[8]}\n")
                        else:
                            print(f"Ignoring feature at line: {line.strip()} as it is removed due to gaps in the fasta file.")
                    else:
                        print(f"Ignoring feature at line: {line.strip()} as it exceeds the length of the corresponding sequence.")
                else:
                    print(f"Ignoring feature at line: {line.strip()} as its sequence ID is not found in the fasta file.")

    # Step 3: Write the gap-removed fasta file
    if save_fasta:
        with open(output_fasta, 'w') as f:
            for seq_id, sequence in fasta_sequences.items():
                f.write(f">{seq_id}\n{sequence}\n")

In [ ]:
path = "./fastas_and_gff/Alignments/"
input_fasta = path+"alignments_all_clean_3.fasta"
input_gff = path+"alignments_all_clean_3.gff"
output_fasta = path+"test_edited_alignments_all_clean_3.fasta"
output_gff = path+"test_edited_alignments_all_clean_3.gff"

#remove_gaps_and_update_gff(input_fasta, input_gff, output_fasta, output_gff, save_fasta = False)

Species genomes (+reversed)

In [ ]:
output_path = "./fastas_and_gff/Alignments/"
path_rev="./fastas_and_gff/reversed_fasta_gtf/"
input_path_main = "/mnt/archive/euglena_genomy/"

In [ ]:
input_path = input_path_main+"gracilis/"
input_fasta = input_path+'gracilis_dbg2olc.fasta'
#output_fasta = output_path+"gracilis_dbg2olc_nogaps.fasta"

remove_gaps_and_update_gff(input_fasta = input_fasta,
                           input_gff = input_path+'gracilis_stringtie_strand_informed.gtf',
                           #output_fasta = output_fasta,
                           #output_gff = output_path+"gracilis_stringtie_strand_informed_nogaps.gtf",
                           save_fasta=True)

remove_gaps_and_update_gff(input_fasta = input_fasta,
                           input_gff = path_rev+"gracilis_stringtie_strand_informed_reversed.gtf",
                        #    output_fasta = output_fasta,
                        #    output_gff = output_path+"gracilis_stringtie_strand_informed_reversed_nogaps.gtf",
                           save_fasta=False)

In [ ]:
input_path = input_path_main+"hiemalis/"
input_fasta = input_path+'hiemalis_rascaf.fasta'
output_fasta = output_path+"hiemalis_rascaf_nogaps.fasta"

remove_gaps_and_update_gff(input_fasta = input_fasta,
                           input_gff = input_path+'hiemalis_stringtie_strand_informed3.gtf',
                           output_fasta = output_fasta,
                           output_gff = output_path+"hiemalis_stringtie_strand_informed3_nogaps.gtf",
                           save_fasta=True)

remove_gaps_and_update_gff(input_fasta = input_fasta,
                           input_gff = path_rev+"hiemalis_stringtie_strand_informed3_reversed.gtf",
                           output_fasta = output_fasta,
                           output_gff = output_path+"hiemalis_stringtie_strand_informed3_reversed_nogaps.gtf",
                           save_fasta=False)

In [ ]:
input_path = input_path_main+"longa/"
input_gff = input_path+'longa_stringtie_strand_informed.gtf'
input_fasta = input_path+'longa_rascaf.fasta'
output_fasta = output_path+"longa_rascaf_nogaps.fasta"



remove_gaps_and_update_gff(input_fasta = input_fasta,
                           input_gff = input_path+'longa_stringtie_strand_informed.gtf',
                           output_fasta = output_fasta,
                           output_gff = output_path+"longa_stringtie_strand_informed_nogaps.gtf",
                           save_fasta=True)

remove_gaps_and_update_gff(input_fasta = input_fasta,
                           input_gff = path_rev+"longa_stringtie_strand_informed_reversed.gtf",
                           output_fasta = output_fasta,
                           output_gff = output_path+"longa_stringtie_strand_informed_reversed_nogaps.gtf",
                           save_fasta=False)

In [ ]:
input_path = "./fastas_and_gff/"
input_fasta = input_path+'bugtest.fasta'
output_fasta = output_path+"bugtest_nogaps.fasta"

'''
genom_bugtest = './fastas_and_gff/bugtest.fasta'
geny_bugtest = './fastas_and_gff/bugtest.gtf'
geny_bugtest_rev = './fastas_and_gff/bugtest_reversed.gtf'
'''

remove_gaps_and_update_gff(input_fasta = input_fasta,
                           input_gff = input_path+'bugtest.gtf',
                           output_fasta = output_fasta,
                           output_gff = output_path+"bugtest_nogaps.gtf",
                           save_fasta=True)

remove_gaps_and_update_gff(input_fasta = input_fasta,
                           input_gff = path_rev+"bugtest_reversed.gtf",
                           output_fasta = output_fasta,
                           output_gff = output_path+"bugtest_reversed_nogaps.gtf",
                           save_fasta=False)